In [8]:
from llama_index.core import Settings
from llama_index.llms.groq import Groq

In [9]:
import os
llm = Groq(model="llama-3.3-70b-versatile",
           api_key=os.environ.get("GROQ_API_KEY"))

In [10]:
def calcular_imposto_renda(rendimento: float) -> str:
    """
    Calcula o imposto de renda com base no rendimento anual.
    Args:
        rendimento (float): Rendimento anual

    Returns:
        str: O valor do imposto devido com base no rendimento
    """
    if rendimento <= 2000:
        return "Você está isento de pagar imposto de renda"
    elif 2000 < rendimento <= 5000:
        imposto = (rendimento - 2000) * 0.10
        return f"O imposto devido é de R$ {imposto:.2f}, base em um rendimento de R$ {rendimento:.2f}"
    elif 5000 < rendimento <= 10000:
        imposto = (rendimento - 5000) * 0.15 + 300
        return f"O imposto devido é de R$ {imposto:.2f}, base em um rendimento de R$ {rendimento:.2f}"
    else:
        imposto = (rendimento - 10000) * 0.20 + 1050
        return f"O imposto devido é de R$ {imposto:.2f}, base em um rendimento de R$ {rendimento:.2f}"

### Converterndo função em ferramenta

In [11]:
from llama_index.core.tools import FunctionTool

In [12]:
ferramenta_imposto_renda = FunctionTool.from_defaults(
    fn=calcular_imposto_renda,
    name="calcular_imposto_renda",
    description="Calcula o imposto de renda com base no rendimento anual."
                "Argummento: rendimento (float)."
                "Retorna o valor do imposto devido de acordo com faixa de rendimento."
)

In [13]:
from llama_index.core.agent import FunctionCallingAgentWorker

In [14]:
agent_worker_imposto = FunctionCallingAgentWorker.from_tools(
    tools=[ferramenta_imposto_renda],
    verbose=True,
    allow_parallel_tool_calls=True,
    llm=llm
)

In [15]:
from llama_index.core.agent import AgentRunner

In [16]:
agente_imposto = AgentRunner(agent_worker_imposto)

C:\Users\DELL\AppData\Local\Temp\ipykernel_1888\422229654.py:1: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  agente_imposto = AgentRunner(agent_worker_imposto)


In [17]:
response = agente_imposto.chat("""
                               Qual é o imposto de renda devido por uma pessoa com rendimento anual de R$ 7500?
                               """)


Added user message to memory: 
                               Qual é o imposto de renda devido por uma pessoa com rendimento anual de R$ 7500?
                               
=== Calling Function ===
Calling function: calcular_imposto_renda with args: {"rendimento": 7500}
=== Function Output ===
O imposto devido é de R$ 675.00, base em um rendimento de R$ 7500.00
=== LLM Response ===
O imposto de renda devido por uma pessoa com rendimento anual de R$ 7500 é de R$ 675,00.


In [18]:
import arxiv

def consulta_artigos(title: str) -> str:
    """Consulta os artigos na base Arxiv e retorna resultados formatados"""
    busca = arxiv.Search(
        query=title, 
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance
    )
    resultados = [
        f"Artigo: {artigo.title}\n"
        f"Categoria: {artigo.primary_category}\n"
        f"Link: {artigo.entry_id}\n"
        for artigo in busca.results()
    ]
    return "\n\n".join(resultados)

In [19]:
consulta_artigos_tool = FunctionTool.from_defaults(fn=consulta_artigos)

In [20]:
agent_worker = FunctionCallingAgentWorker.from_tools(
    tools=[ferramenta_imposto_renda, consulta_artigos_tool],
    verbose=True,
    allow_parallel_tool_calls=True,
    llm=llm
)

In [21]:
agent = AgentRunner(agent_worker)
response = agent.chat("Me retorne artigos sobre LangChain na educação")

Added user message to memory: Me retorne artigos sobre LangChain na educação


C:\Users\DELL\AppData\Local\Temp\ipykernel_1888\496446454.py:1: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  agent = AgentRunner(agent_worker)


=== Calling Function ===
Calling function: consulta_artigos with args: {"title": "LangChain na educa\u00e7\u00e3o"}


C:\Users\DELL\AppData\Local\Temp\ipykernel_1888\717003510.py:14: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for artigo in busca.results()


=== Function Output ===
Encountered error: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=LangChain+na+educa%C3%A7%C3%A3o&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)
=== Calling Function ===
Calling function: consulta_artigos with args: {"title": "LangChain education"}
=== Function Output ===
Encountered error: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=LangChain+education&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)
=== LLM Response ===
Desculpe, mas não consegui encontrar artigos sobre LangChain na educação devido a um erro de limite de requisições. Você pode tentar novamente mais tarde ou procurar em outras fontes.


### Usando o Tavily

In [22]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [23]:
tavily_key = os.environ.get("TAVILY_API_KEY")

In [24]:
from llama_index.tools.tavily_research import TavilyToolSpec
tavily_tool = TavilyToolSpec(
    api_key=tavily_key
)

In [25]:
tavily_tool_list = tavily_tool.to_tool_list()
for tool in tavily_tool_list:
    print(tool.metadata.name)

search


In [26]:
tavily_tool.search("Me retorne artigos científicos sobre LangChain", max_results=3)

[Document(id_='f53bdc3f-13e3-43f3-803b-cdcff3cba13a', embedding=None, metadata={'url': 'https://medium.com/@recogna/desmistificando-langchain-a1b397bc8fb0'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='# Introdução ao LangChain. S e você é apaixonado por Inteligência… | by Recogna | Medium\n\nSitemap\n\nOpen in app\n\nSign up\n\nSign in\n\n e Processamento de Linguagem Natural (PLN), veio ao lugar certo. Hoje iniciamos uma série de artigos sobre LangChain, uma ferramenta revolucionária que está transformando a maneira como interagimos com os Large Language Models (LLMs, os Grandes Modelos de Linguagem). Neste primeiro artigo, exploraremos o que é LangChain e como ele gerencia a importação de modelos, um aspecto crucial para aqueles que desejam iniciar a exploração dessa poderosa tecnologia.\n\nNeste artigo, abordaremos os segui

In [27]:
from llama_index.core.tools import FunctionTool
tavily_tool_function = FunctionTool.from_defaults(
    fn=tavily_tool.search,
    name="TavilySearch",
    description="Busca artigos com Tavily sobre determinado tópico."
)

In [28]:
agent_worker = FunctionCallingAgentWorker.from_tools(
    tools=[tavily_tool_function],
    verbose=True,
    allow_parallel_tool_calls=False,
    llm=llm
)

In [29]:
agent = AgentRunner(agent_worker)

C:\Users\DELL\AppData\Local\Temp\ipykernel_1888\3302559522.py:1: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  agent = AgentRunner(agent_worker)


In [30]:
response = agent.chat("Me retorne artigos sobre LLM e LangChain")

Added user message to memory: Me retorne artigos sobre LLM e LangChain
=== Calling Function ===
Calling function: TavilySearch with args: {"max_results": 10, "query": "LLM e LangChain"}
=== Function Output ===
[Document(id_='831ece60-a687-4244-9bb9-f9be6bc6a139', embedding=None, metadata={'url': 'https://blog.dsacademy.com.br/langchain-aplicacoes-de-inteligencia-artificial-generativa-com-llms'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='LangChain é um framework de código aberto (open-source) criado para simplificar o desenvolvimento de aplicativos usando Large Language Models (LLMs), como aqueles disponibilizados por OpenAI ou Hugging Face. Isso permite que você crie aplicativos dinmicos e responsivos a dados que aproveitam os avanços mais recentes no Processamento de Linguagem Natural.\n\nSeja você um Engenheiro de IA, um Ci

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.3-70b-versatile` in organization `org_01k892tth3e80rbw27b8epeht6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12502, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [31]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex


In [32]:
url = "files/LLM.pdf"
artigo = SimpleDirectoryReader(input_files=[url]).load_data()

Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 155 0 (offset 0)
Ignoring wrong pointing object 158 0 (offset 0)
Ignoring wrong pointing object 160 0 (offset 0)
Ignoring wrong pointing object 163 0 (offset 0)
Ignori

In [33]:
url = "files/LLM_2.pdf"
tutorial = SimpleDirectoryReader(input_files=[url]).load_data()

### Gerar os Embeddings

In [34]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [35]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name = "intfloat/multilingual-e5-large"
)

In [36]:
artigo_index = VectorStoreIndex.from_documents(artigo)
tutorial_index = VectorStoreIndex.from_documents(tutorial)

In [37]:
artigo_index.storage_context.persist(persist_dir="artigo")
tutorial_index.storage_context.persist(persist_dir="tutorial")

### Engine de busca

In [38]:
from llama_index.core import StorageContext, load_index_from_storage

In [39]:
storage_context = StorageContext.from_defaults(persist_dir="artigo")
artigo_context = load_index_from_storage(storage_context)
storage_context = StorageContext.from_defaults(persist_dir="tutorial")
tutorial_context = load_index_from_storage(storage_context)

Loading llama_index.core.storage.kvstore.simple_kvstore from artigo\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from artigo\index_store.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from tutorial\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from tutorial\index_store.json.


In [41]:
artigo_engine = artigo_context.as_query_engine(similarity_top_k=3, llm=llm)
tutorial_engine = tutorial_context.as_query_engine(similarity_top_k=3, llm=llm)

In [42]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

In [43]:
query_engine_tool = [
    QueryEngineTool(
        query_engine=artigo_engine,
        metadata=ToolMetadata(
            name="artigo_engine",
            description=(
                "Fornece informações sobre LLM e LangChain."
                "Use uma ferramenta detalhada em texto simples como entrada para a ferramenta")
        )
    ),
    QueryEngineTool(
        query_engine=tutorial_engine,
        metadata=ToolMetadata(
            name="tutorial_engine",
            description=(
                "Fornece informações sobre casos de uso e aplicações em LLMs"
                "Use uma pergunta detalhada em texto simples como entrada para a ferramenta")
        )
    )
]

In [45]:
agent_worker = FunctionCallingAgentWorker.from_tools(
    query_engine_tool,
    verbose=True,
    allow_parallel_tool_calls=True,
    llm=llm
)

agent_document = AgentRunner(agent_worker)

C:\Users\DELL\AppData\Local\Temp\ipykernel_1888\1811318807.py:8: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  agent_document = AgentRunner(agent_worker)


In [46]:
response = agent_document.chat("Quais são as aplicações posso construir com LLM e LangChain?")

Added user message to memory: Quais são as aplicações posso construir com LLM e LangChain?
=== Calling Function ===
Calling function: tutorial_engine with args: {"input": "Quais s\u00e3o as aplica\u00e7\u00f5es que posso construir com LLM e LangChain?"}
=== Function Output ===
As aplicações que você pode construir com LLM e LangChain incluem:

1. Criação e aprimoramento de conteúdo, como geração automática de texto, assistência na redação, tradução automática, resumo de textos e planejamento de conteúdo.
2. Análise e organização de informações, como análise de sentimento, extração de informações, classificação de textos e revisão técnica.
3. Interação e automação, como chatbots, perguntas e respostas, e geração de respostas a perguntas com base em um corpus.
4. Aplicações multimodais, como geração de conteúdo audiovisual, interpretação de dados de imagens, tradução de conteúdo multimídia e criação de experiências interativas ricas.

Além disso, você pode construir aplicações práticas, 

In [47]:
response = agent_document.chat("Quais as principais tendências em LLM e LangChain?")

Added user message to memory: Quais as principais tendências em LLM e LangChain?
=== Calling Function ===
Calling function: tutorial_engine with args: {"input": "Quais s\u00e3o as principais tend\u00eancias em LLM e LangChain?"}
=== Function Output ===
As principais tendências em LLM incluem a proliferação de modelos de código aberto, como o Llama, Vicuna, Falcon, Mistral e Gemma, que democratizam o acesso à tecnologia de ponta de processamento de linguagem. Além disso, a integração de LLMs às ferramentas de desenvolvimento de software e de escritório está transformando a eficiência e a capacidade das empresas. A Microsoft e o Google já integraram LLMs em seus pacotes de produtividade, como o Microsoft 365 Copilot e o Google Workspace.

Em relação às tipologias de LLM, os modelos baseados em transformers são a arquitetura dominante atualmente, permitindo que os modelos capturem estruturas gramaticais complexas e dependências de palavras com longa distância. A série de modelos GPT desen

### Agente ReAct

In [48]:
from llama_index.core.agent import ReActAgent

In [51]:
agent = ReActAgent.from_tools(
    query_engine_tool,
    verbose=True,
    llm=llm
)

c:\Users\DELL\Documents\cursos\llamaindex-agents\.venv\Lib\site-packages\llama_index\core\agent\react\base.py:154: DeprecationWarning: Call to deprecated class ReActAgent. (ReActAgent has been rewritten and replaced by llama_index.core.agent.workflow.ReActAgent.

This implementation will be removed in a v0.13.0 and the new implementation will be promoted to the `from llama_index.core.agent import ReActAgent` path.

See the docs for more information: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return cls(
c:\Users\DELL\Documents\cursos\llamaindex-agents\.venv\Lib\site-packages\deprecated\classic.py:184: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return old_new1(cls, *args, **kwargs)


In [52]:
response = agent.chat("Quais são principais ferramentas usadas com LangChain?")

> Running step 0e9e9db0-b1b7-4861-b57b-0927e83d05bb. Step input: Quais são principais ferramentas usadas com LangChain?
Thought: O usuário está perguntando sobre as principais ferramentas usadas com LangChain. Eu preciso usar uma ferramenta para fornecer uma resposta detalhada.
Action: artigo_engine
Action Input: {'input': 'ferramentas usadas com LangChain'}
Observation: Não há menção a ferramentas usadas com LangChain no texto fornecido. No entanto, são mencionadas ferramentas como MLflow e Hugging Face, que são usadas em conjunto com modelos de linguagem, como os LLMs. Além disso, a Databricks é mencionada como uma plataforma que fornece ferramentas integradas para usar e ajustar os LLMs.
> Running step 9d766cc8-750b-46e0-812f-b2417d362beb. Step input: None
Thought: A ferramenta artigo_engine não forneceu informações específicas sobre as principais ferramentas usadas com LangChain, mas mencionou algumas ferramentas relacionadas a modelos de linguagem, como MLflow, Hugging Face e Data

In [53]:
response = agent.chat("Quais as principais tendências em LangChain que eu deveria estudar?")

> Running step 9c82b61d-a5d5-4ce5-9e2e-25939af14529. Step input: Quais as principais tendências em LangChain que eu deveria estudar?
Thought: O usuário está procurando por tendências em LangChain. Para fornecer uma resposta mais precisa, devo usar a ferramenta tutorial_engine.
Action: tutorial_engine
Action Input: {'input': 'Quais as principais tendências em LangChain?'}
Observation: As principais tendências em LangChain incluem a proliferação de LLMs de código aberto, que democratizou o acesso à tecnologia de ponta de processamento de linguagem, permitindo que pesquisadores, desenvolvedores e amadores experimentassem, personalizassem e implantassem soluções de IA com um investimento inicial mínimo. Além disso, a integração do LLM às ferramentas de desenvolvimento de software e de escritório está transformando a eficiência e a capacidade das empresas. Outra tendência é a evolução dos LLMs para além da simples previsão de texto, tornando-se aplicativos sofisticados em vários domínios, a